# 03 - Logistic Regression Baseline

A reproducible baseline for detecting pump windows from market activity features. Standardisation is inside the pipeline so validation-fold information cannot leak into training.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'labeled_features').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'labeled_features' / 'features_25S.csv.gz'
FEATURES = ['std_rush_order', 'avg_rush_order', 'std_trades', 'std_volume', 'avg_volume', 'std_price', 'avg_price', 'avg_price_max', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos']

## Load data and define the model

We use `class_weight='balanced'` because only a tiny fraction of windows are labeled as pumps.

In [2]:
data = pd.read_csv(DATA_PATH)
X = data[FEATURES]
y = data['gt'].astype(int)
print(f'Observations: {len(data):,}')
print(f'Pump windows: {y.sum():,} ({y.mean():.3%})')

model = Pipeline([
    ('scale', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=2_000, class_weight='balanced', random_state=42)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Observations: 482,157
Pump windows: 317 (0.066%)


## Stratified five-fold validation

In [3]:
probability = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
prediction = (probability >= 0.5).astype(int)

print(classification_report(y, prediction, target_names=['normal', 'pump'], digits=3))
print('Confusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y, prediction))
print(f'ROC-AUC: {roc_auc_score(y, probability):.3f}')
print(f'Average precision: {average_precision_score(y, probability):.3f}')

              precision    recall  f1-score   support

      normal      1.000     0.999     0.999    481840
        pump      0.335     0.984     0.500       317

    accuracy                          0.999    482157
   macro avg      0.667     0.991     0.749    482157
weighted avg      1.000     0.999     0.999    482157

Confusion matrix [[TN, FP], [FN, TP]]:
[[481220    620]
 [     5    312]]
ROC-AUC: 0.995
Average precision: 0.905


Expected result on the supplied 25-second data: recall about **0.984**, ROC-AUC about **0.995**, and average precision about **0.905**. Because the data is highly imbalanced, focus discussion on recall and precision rather than the very high accuracy.